In [ ]:
BASE_PATH = '/kaggle/input/llm-classification-finetuning'
TRAIN_DATA = BASE_PATH + '/train.csv'
TEST_DATA = BASE_PATH + '/test.csv'

In [ ]:
ENABLE_SUBMISSION = True

## Load dataset

In [ ]:
import pandas as pd
df = pd.read_csv(TRAIN_DATA)
test_df = pd.read_csv(TEST_DATA)

# Change lable to 0, 1, 2

In [ ]:
def make_label(row):
    if row['winner_model_a'] == 1:
        return 0
    elif row['winner_model_b'] == 1:
        return 1
    else:
        return 2


df["label"] = df.apply(make_label, axis=1)

In [ ]:
def extract_text(text):
    return text[2:-2]

df["prompt-extracted"] = df['prompt'].apply(extract_text)
df["response_a-extracted"] = df['response_a'].apply(extract_text)
df["response_b-extracted"] = df['response_b'].apply(extract_text)

# Preprocessing data

In [ ]:
import re
import string

def clean_text(text):
    text = text.lower() # lower case
    text = re.sub(r"\[.*?\]", "", text)  # loại bỏ dấu [..]
    text = re.sub(r"\s+", " ", text).strip()
    return text

for col in ["prompt-extracted", "response_a-extracted", "response_b-extracted"]:
    df[col] = df[col].apply(clean_text)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df["label"] = df["label"].fillna(2)

In [ ]:
# get main_df
main_df = df[['id', 'prompt-extracted', 'response_a-extracted', 'response_b-extracted', 'label']]

In [ ]:
main_df

In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_val = train_test_split(main_df, test_size=0.1, stratify=df["label"], random_state=42)

## Bert Base uncased

In [ ]:
# MODEL_PATH="bert-base-uncased"
MODEL_PATH="/kaggle/input/transformers/bert-base-uncased"

## Tokenize

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

def tokenize_fn(example):
    # Combine prompt + response A + response B for each row in the batch
    texts = [
        p + tokenizer.sep_token + a + tokenizer.sep_token + b
        for p, a, b in zip(
            example["prompt-extracted"],
            example["response_a-extracted"],
            example["response_b-extracted"]
        )
    ]
    
    return tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=256,
    )

In [ ]:
# prepare dataset for training
from datasets import Dataset
import torch

train_dataset = Dataset.from_pandas(df_train)
val_dataset = Dataset.from_pandas(df_val)

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)

train_dataset = train_dataset.remove_columns(df_train.columns).with_format("torch")
val_dataset = val_dataset.remove_columns(df_val.columns).with_format("torch")

train_dataset = train_dataset.add_column("labels", df_train["label"].tolist())
val_dataset = val_dataset.add_column("labels", df_val["label"].tolist())

In [ ]:
train_dataset

In [ ]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=3)

In [ ]:
# Train
from transformers import TrainingArguments, Trainer

from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted")
    }


training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
trainer.evaluate()

## Test 

In [ ]:
# Create test dataset
test_df["prompt-extracted"] = test_df['prompt'].apply(extract_text)
test_df["response_a-extracted"] = test_df['response_a'].apply(extract_text)
test_df["response_b-extracted"] = test_df['response_b'].apply(extract_text)

test_dataset = Dataset.from_pandas(test_df)
test_dataset = test_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.remove_columns(test_df.columns).with_format("torch")

predictions = trainer.predict(test_dataset)
probs = torch.nn.functional.softmax(torch.tensor(predictions.predictions), dim=-1)

In [ ]:
probs

In [ ]:
test_df['id']

In [ ]:
if ENABLE_SUBMISSION:
    test_ids = test_df['id']
    submission = pd.DataFrame({
            "id": test_ids,
            "winner_model_a": probs[:, 0],
            "winner_model_b": probs[:, 1],
            "winner_tie": probs[:, 2]
        })
    submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("✅ Submission file saved: submission.csv")